# 2. Čišćenje i pretprocesiranje tekstualnih podataka
## Nakon inicijalne analize podataka, proces čišćenja i pretprocesiranja tekstualnih podataka prilagođen je specifičnim karakteristikama skupa podataka. Tokom analize uočene su razlike između istinitih i lažnih vesti ne samo u samom sadržaju teksta, već i u stilu pisanja, prisustvu URL adresa, upotrebi velikih slova i interpunkcijskih znakova. Zbog toga nije primenjena agresivna normalizacija teksta, kako se ne bi izgubile potencijalno važne informacije za klasifikaciju.

## 2.1 Učitavanje i spajanje skupova

In [2]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

true_df = pd.read_csv('data/true.csv')
fake_df = pd.read_csv('data/fake.csv')

true_df['label'] = 1
fake_df['label'] = 0

news_df = pd.concat([true_df, fake_df], ignore_index=True)
news_df = news_df.sample(frac=1, random_state=42).reset_index(drop=True)

## 2.2 Uklanjanje duplikata

In [3]:
news_df = news_df.drop_duplicates()
news_df = news_df.drop_duplicates(subset=['text'])

## 2.3 Uklanjanje kratkih tekstova

In [4]:
news_df['word_count'] = news_df['text'].astype(str).apply(lambda x: len(x.split()))
news_df = news_df[news_df['word_count'] >= 10]

## 2.4 Formiranje jedinstvenog tekstualnog sadržaja

In [5]:
news_df['content'] = (news_df['title'].astype(str) + ' ' + news_df['text'].astype(str))

## 2.5 Izdvajanje stilskih karakteristika

In [6]:
news_df['has_url'] = news_df['content'].str.contains(r'http[s]?://|www\\.', regex=True)
news_df['exclamation_count'] = (news_df['content'].str.count('!'))
news_df['caps_ratio'] = news_df['content'].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
news_df['text_length'] = (news_df['content'].apply(len))
news_df['word_count'] = (news_df['content'].apply(lambda x: len(x.split())))

## 2.6 Čišćenje teksta

In [7]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9!?.,:/\s]', '', text)

    return text.strip()

news_df['clean_content'] = (news_df['content'].apply(clean_text))

## 2.7 Tokenizacija teskta

In [8]:
nltk.download('punkt')
nltk.download('punkt_tab')
news_df['tokens'] = news_df['clean_content'].apply(str.split)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Saska\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Saska\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 2.8 Uklanjanje reči koje nemaju značajnu informativnu vrednost

In [9]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
news_df['tokens'] = news_df['tokens'].apply(lambda words: [word for word in words if word.lower() not in stop_words])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Saska\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 2.9 Lematizacija

In [10]:
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
news_df['tokens'] = news_df['tokens'].apply(lambda words: [lemmatizer.lemmatize(word) for word in words])

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Saska\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 2.10 Formiranje finalnog teksta

### Nakon pretprocesiranja, tokeni su ponovo spojeni u tekstualni oblik.

In [11]:
news_df['final_text'] = (news_df['tokens'].apply(lambda words: ' '.join(words)))

print(news_df)

                                                   title  \
0       BREAKING: GOP Chairman Grassley Has Had Enoug...   
1       Failed GOP Candidates Remembered In Hilarious...   
2       Mike Pence’s New DC Neighbors Are HILARIOUSLY...   
3      California AG pledges to defend birth control ...   
4      AZ RANCHERS Living On US-Mexico Border Destroy...   
...                                                  ...   
44891  Fighting in southern Philippine city may end i...   
44892  New York protesters camp out at Goldman Sachs ...   
44895  ATHEISTS SUE GOVERNOR OF TEXAS Over Display on...   
44896  Republican tax plan would deal financial hit t...   
44897  U.N. refugee commissioner says Australia must ...   

                                                    text          subject  \
0      Donald Trump s White House is in chaos, and th...             News   
1      Now that Donald Trump is the presumptive GOP n...             News   
2      Mike Pence is a huge homophobe. He suppor

In [12]:
news_df.to_csv('data/preprocessed_news.csv', index=False)
